# 4.0 — Stability across random seeds

In [ ]:
import sys, pathlib
SRC = pathlib.Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm

from paths import CLEAN_DATA, INTERIM_DATA
from metrics import total_return, ann_active_return
from chart import chart


# Load Data

In [ ]:
df_merge = pd.read_csv(CLEAN_DATA / '10y_merged.csv')
df_merge = df_merge.sort_values('date').set_index('date')


In [ ]:
df_factors = pd.read_csv(INTERIM_DATA / '10y_factors.csv')

factors = ['P/E', 'P/B', 'P/S', 'EV/EBITDA', 'FCF Yield', 'Earnings Yield']

df_factors[factors] = df_factors[factors].astype(float)
df_factors[factors] = stats.zscore(df_factors[factors])
zscore = df_factors.drop(columns=['12 Mo Yield', 'Company Id', 'SecId']).copy()
zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']] = - zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']]


# Train / Test Split

In [ ]:
_n_ = 50

split = int(_n_ / 100 * len(df_merge))

df_train = df_merge.iloc[:split]
df_test  = df_merge.iloc[split:]

bmk_train = df_train['sprtrn']
bmk_test  = df_test['sprtrn']

rf_train = df_train['rf']
rf_test  = df_test['rf']


# Stability across random seeds

Wrap the full pipeline in a function and rerun it many times with different seeds. Plot total and active returns vs. run index with an OLS fit to confirm the slope is approximately zero (results are stable).

In [ ]:
def run_model_once(random_state=None):

    if random_state is not None:
        np.random.seed(random_state)

    port_size    = 45
    trials_train = 10**4
    port_train   = 10**3
    top_k        = 50


    # Train

    best_obj_train = -1e9
    best_weight    = None
    best_tic       = None
    best_ret_train = None

    df_m = df_train.copy()
    model_list = []

    for i in range(trials_train):
        z_var = zscore.copy()

        onelist = np.ones(len(factors))
        weight = np.random.dirichlet(onelist)
        weight = weight / weight.sum()

        z_var[factors] = z_var[factors].mul(weight, axis=1)
        z_var["score"] = z_var[factors].sum(axis=1)

        z_var = z_var.sort_values("score", ascending=False).reset_index(drop=True)

        z_var_tic = z_var["Ticker"].iloc[:port_size]

        port_ret = df_m[z_var_tic].mean(axis=1)

        obj = ann_active_return(port_ret, bmk_train)

        model_list.append((obj, weight.copy()))

        if obj > best_obj_train:
            best_obj_train = obj
            best_weight    = weight.copy()
            best_tic       = z_var_tic.copy()
            best_ret_train = port_ret.copy()

    # average across top K factor models
    model_list.sort(key=lambda x: x[0], reverse=True)
    top_models  = model_list[:top_k]

    avg_weight = np.mean([w for _, w in top_models], axis=0)
    avg_weight = avg_weight / avg_weight.sum()

    best_weight = avg_weight.copy()

    # recompute tickers and train return using averaged factor weights
    z_var = zscore.copy()
    z_var[factors] = z_var[factors].mul(best_weight, axis=1)
    z_var["score"] = z_var[factors].sum(axis=1)
    z_var = z_var.sort_values("score", ascending=False).reset_index(drop=True)

    best_tic = z_var["Ticker"].iloc[:port_size]
    df_m = df_train.copy()
    best_ret_train = df_m[best_tic].mean(axis=1)
    best_obj_train = ann_active_return(best_ret_train, bmk_train)

    # stock weight optimisation

    best_obj_w   = -1e9
    best_stock_w = None

    tickers = list(best_tic)
    df_m = df_train.copy()
    df_m = df_m[tickers + ["sprtrn"]]

    for j in range(port_train):
        onelist = np.ones(len(tickers))
        stock_w = np.random.dirichlet(onelist)
        stock_w = stock_w / stock_w.sum()

        port_ret = df_m[tickers].mul(stock_w, axis=1).sum(axis=1)

        obj = ann_active_return(port_ret, bmk_train)

        if obj > best_obj_w:
            best_obj_w     = obj
            best_stock_w   = stock_w.copy()
            best_ret_train = port_ret.copy()

    df_final_train = pd.DataFrame({"Ticker": best_tic, "Weight": best_stock_w})

 
    # Test
   
    df_m_test = df_test.copy()

    port_tic = df_m_test[df_final_train["Ticker"]].copy()

    w_test = df_final_train["Weight"].astype(float).to_numpy()
    best_ret_test = port_tic.mul(w_test, axis=1).sum(axis=1)

    best_obj_test = ann_active_return(best_ret_test, bmk_test)

    # returns dict so we can inspect whatever we want
    return {
        "best_obj_train": best_obj_train,
        "best_obj_w": best_obj_w,
        "best_obj_test": best_obj_test,                 # annual active return on test
        "test_total": total_return(best_ret_test),      # total return of portfolio on test
        "train_total": total_return(best_ret_train),
        "tickers": best_tic,
        "weights": best_stock_w,
    }


In [ ]:
n_runs = 50          # how many times you want to rerun the whole thing
test_totals  = []
test_active  = []

for r in range(n_runs):
    stats = run_model_once(random_state=1000 + r)   # different seed each time
    test_totals.append(stats["test_total"])
    test_active.append(stats["best_obj_test"])

In [ ]:
runs = np.arange(1, n_runs + 1)
bench_total = total_return(bmk_test)    # decimal

y_tot = np.array(test_totals) * 100          # total return in percent
y_act = np.array(test_active)               # active return already in percent

# OLS fit for total return
slope_tot, intercept_tot = np.polyfit(runs, y_tot, 1)
x_line = np.linspace(runs.min(), runs.max(), 100)
y_line_tot = slope_tot * x_line + intercept_tot

# OLS fit for active return
slope_act, intercept_act = np.polyfit(runs, y_act, 1)
y_line_act = slope_act * x_line + intercept_act

# optional: print slopes to show they are basically zero
print(f"Total return slope: {slope_tot:.4f} percentage points per run")
print(f"Active return slope: {slope_act:.4f} percentage points per run")

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)


ax = axes[0]
ax.scatter(runs, y_tot,
           label="Model total return on test (percent)",
           marker="o")
ax.plot(x_line, y_line_tot,
        label=f"OLS fit (slope {slope_tot:.3f})")
ax.axhline(y=bench_total * 100,
           linestyle="--", color="red",
           label="Benchmark total return (percent)")

ax.set_ylabel("Total return over test window (percent)")
ax.set_title("Stability of model test total return over random searches")
ax.legend()
ax.grid(alpha=0.3)


ax = axes[1]
ax.scatter(runs, y_act,
           label="Annual active return on test (percent)",
           marker="o")
ax.plot(x_line, y_line_act,
        label=f"OLS fit (slope {slope_act:.3f})")
ax.axhline(y=0.0, linestyle="--", color="red",
           label="Benchmark (zero active)")

ax.set_xlabel("Run number")
ax.set_ylabel("Annual active return on test (percent)")
ax.set_title("Stability of model test active return")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

runs = np.arange(1, n_runs + 1)

y_tot = np.array(test_totals) * 100      # total return in percent
y_act = np.array(test_active)           # active return in percent

def test_slope_zero(x, y, label="", alpha=0.10):
    X = sm.add_constant(x)              # add intercept
    model = sm.OLS(y, X).fit()

    slope    = model.params[1]
    slope_se = model.bse[1]
    t_stat   = model.tvalues[1]
    p_val    = model.pvalues[1]

    # alpha = 0.10 gives a 90 percent confidence interval
    ci       = model.conf_int(alpha=alpha)
    ci_low, ci_high = ci[1]             # row 1 is the slope

    level = int((1 - alpha) * 100)

    print(f"{label}")
    print(f"  slope           : {slope:.5f} (percent per run)")
    print(f"  standard error  : {slope_se:.5f}")
    print(f"  t statistic     : {t_stat:.3f}")
    print(f"  p value         : {p_val:.3f}")
    print(f"  {level}% CI       : [{ci_low:.5f}, {ci_high:.5f}]\n")

test_slope_zero(runs, y_tot, "Total return", alpha=0.10)
test_slope_zero(runs, y_act, "Active return", alpha=0.10)
